In [10]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def load(path):
    images = {}
    files = [f for f in os.listdir(path) if f.lower().endswith('.png')]

    for file in files:
        fpath = os.path.join(path, file)
        
        img = cv2.imread(fpath)
        images[file] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # BGR para RGB
            
    return images

set5, set14 = r'.\dataset\Set5', r'.\dataset\Set14'
set5, set14 = load(set5), load(set14)


In [11]:
def part(img):
    r = img[:, :, 0]
    g = img[:, :, 1]
    b = img[:, :, 2]
    channels = (r, g, b)
        
    return channels

In [12]:
def get_coord(h, w, scale_x, scale_y):

    # Calcula novas dimensões baseadas nas escalas
    h = int(h * scale_y)
    w = int(w * scale_x)
    
    # Cria a grade de coordenadas da nova imagem
    y, x = np.mgrid[0:h, 0:w]
    
    # Mapeia de volta para a imagem original
    y = (y + 0.5) / scale_y - 0.5
    x = (x + 0.5) / scale_x - 0.5
    
    return x, y

In [13]:
def nni(channel, scale_x, scale_y):
    h, w = channel.shape
    x, y = get_coord(h, w, scale_x, scale_y)
    
    y_near = np.round(y).astype(int)
    x_near = np.round(x).astype(int)
    
    y_near = np.clip(y_near, 0, h - 1)
    x_near = np.clip(x_near, 0, w - 1)
    
    return channel[y_near, x_near]

In [14]:
def bilinear(channel, scale_x, scale_y):
    h, w = channel.shape
    x, y = get_coord(h, w, scale_x, scale_y)
    
    x0 = np.floor(x).astype(int)
    y0 = np.floor(y).astype(int)

    x1 = np.clip(x0 + 1, 0, w - 1)
    y1 = np.clip(y0 + 1, 0, h - 1)
    
    x0 = np.clip(x0, 0, w - 1)
    y0 = np.clip(y0, 0, h - 1)
    
    dx = x - x0
    dy = y - y0
    
    v00 = channel[y0, x0]
    v01 = channel[y0, x1]
    v10 = channel[y1, x0]
    v11 = channel[y1, x1]
    
    v_top = v00 * (1 - dx) + v01 * dx
    v_bottom = v10 * (1 - dx) + v11 * dx
    
    return v_top * (1 - dy) + v_bottom * dy

In [15]:
def idw(channel, scale_x, scale_y, power=2, return_stats=False):
    h, w = channel.shape
    x, y = get_coord(h, w, scale_x, scale_y)
    
    x0 = np.floor(x).astype(int)
    y0 = np.floor(y).astype(int)

    x1 = np.clip(x0 + 1, 0, w - 1)
    y1 = np.clip(y0 + 1, 0, h - 1)

    x0 = np.clip(x0, 0, w - 1)
    y0 = np.clip(y0, 0, h - 1)
    
    neighbors = [(y0, x0), (y0, x1), (y1, x0), (y1, x1)]
    vals = np.array([channel[ny, nx] for ny, nx in neighbors])

    dists_list = []

    for (ny, nx) in neighbors:
        d = np.sqrt((x - nx)**2 + (y - ny)**2)
        dists_list.append(d)
    dists = np.array(dists_list)
    
    sq_dists = dists ** power
    inv_sq_dists = 1.0 / (sq_dists + 1e-5)
    
    Dj = np.sum(inv_sq_dists, axis=0)       # Dj = soma(d^-p)
    weights = inv_sq_dists / (Dj + 1e-5)    # Pesos normalizados
    V_idw = np.sum(weights * vals, axis=0)  # Resultado Final

    if return_stats: 
        Sj = np.sum(sq_dists, axis=0)   # Sj = soma(d^2)
        sum_vals = np.sum(vals, axis=0) # Sum(vals) para calcular a média
        
        stats = {
            'Dj': Dj,
            'Sj': Sj,
            'sum_vals': sum_vals,
            'n_neighbors': 4
        }
        return V_idw, stats
    
    return V_idw

In [16]:
def idwr(channel, scale_x, scale_y):

    V_idw, stats = idw(channel, scale_x, scale_y, power=2, return_stats=True)
    
    Dj, Sj = stats['Dj'], stats['Sj']
    sum_vals, n = stats['sum_vals'],  stats['n_neighbors']
    
    u = sum_vals / n # Calcula a média u
    
    # Correção = (u / (n^2 - Dj*Sj)) * (sum(v_i) - n*V_idw)
    numerator = u * (sum_vals - n * V_idw)
    denominator = (n**2) - (Dj * Sj)
    
    correction = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=np.abs(denominator) > 1e-5)
    
    return V_idw + correction

In [17]:
def process(channels, f, scale_x, scale_y):
    r, g, b = channels

    new_r = f(r, scale_x, scale_y)
    new_g = f(g, scale_x, scale_y)
    new_b = f(b, scale_x, scale_y)
    
    img_final = np.stack([new_r, new_g, new_b], axis=-1)
    return np.clip(img_final, 0, 255).astype(np.uint8)